# 04 — Model Sentimen (Pretrained HuggingFace)
**SuaraLens** | Analisis sentimen menggunakan `w11wo/indonesian-roberta-base-sentiment-classifier`.


In [ ]:
# Install jika belum ada
import subprocess, sys
for pkg in ['transformers', 'torch']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Dependencies OK')


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from modules.sentiment_model import classify_sentiment, classify_sentiment_batch, load_sentiment_model

sns.set_theme(style='whitegrid')
DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/sentiment_results.json'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset: {len(df):,} baris')


## 1. Load Model

In [ ]:
pipe = load_sentiment_model()
print('Model siap digunakan.')


## 2. Uji pada 30 Sampel Proporsional

In [ ]:
# Ambil 10 per kelas sentimen (proporsional)
sample_parts = []
for sent in ['negative', 'neutral', 'positive']:
    part = df[df['sentiment_true'] == sent].sample(min(10, (df['sentiment_true'] == sent).sum()), random_state=42)
    sample_parts.append(part)

sample_df = pd.concat(sample_parts).reset_index(drop=True)
print(f'Sampel: {len(sample_df)} baris ({dict(sample_df["sentiment_true"].value_counts())})')
print('\nMenjalankan inferensi...')

texts    = sample_df['teks_aduan'].tolist()
preds    = classify_sentiment_batch(texts)

sample_df['sentiment_pred']  = [p['sentiment'] for p in preds]
sample_df['sentiment_score'] = [p['score']     for p in preds]


## 3. Hasil vs Ground Truth

In [ ]:
display(sample_df[['id_aduan', 'teks_aduan', 'sentiment_true', 'sentiment_pred', 'sentiment_score']].head(20))

print('\n=== Classification Report (30 sampel — sanity check awal) ===')
print(classification_report(
    sample_df['sentiment_true'],
    sample_df['sentiment_pred'],
    target_names=['negative', 'neutral', 'positive']
))
print('⚠ CATATAN: Ini sanity check awal. Evaluasi formal (NB 05) menggunakan 500 sampel.')


## 4. Analisis: Informal vs Formal

In [ ]:
# Kanal informal vs formal
informal_channels = ['WhatsApp', 'Medsos', 'Instagram']
formal_channels   = ['Email', 'Web Form', 'Surat']

df_sample_kanal = sample_df.copy()
df_sample_kanal['kanal_type'] = df_sample_kanal['kanal'].apply(
    lambda k: 'Informal' if k in informal_channels else
              'Formal'   if k in formal_channels   else 'Lainnya'
)

print('=== Akurasi per Tipe Kanal ===')
for kanal_type in ['Informal', 'Formal', 'Lainnya']:
    subset = df_sample_kanal[df_sample_kanal['kanal_type'] == kanal_type]
    if len(subset) == 0:
        continue
    acc = (subset['sentiment_true'] == subset['sentiment_pred']).mean()
    print(f'  {kanal_type} (n={len(subset)}): Accuracy = {acc*100:.1f}%')

print()
print('Catatan: teks informal (singkatan, ALL CAPS, emoji) mungkin menurunkan akurasi.')
print('Hasil ini adalah indikasi awal — perlu sampel lebih besar untuk kesimpulan definitif.')


## 5. Simpan Hasil ke JSON

In [ ]:
results = []
for _, row in sample_df.iterrows():
    results.append({
        'id_aduan':        row['id_aduan'],
        'sentiment_true':  row['sentiment_true'],
        'sentiment_pred':  row['sentiment_pred'],
        'sentiment_score': row['sentiment_score'],
        'kanal':           row.get('kanal'),
    })

output = {
    'generated_at':  pd.Timestamp.now().isoformat(),
    'model_used':    'w11wo/indonesian-roberta-base-sentiment-classifier',
    'sample_size':   len(results),
    'results':       results,
    'caveats': [
        'Label sentiment_true adalah label sintetis dari generator data, bukan anotasi manusia.',
        'Evaluasi formal dilakukan di NB 05 dengan 500 sampel.',
        'Performa pada teks informal belum tervalidasi dengan dataset representatif.',
    ]
}

output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'Hasil disimpan ke: {OUTPUT_PATH}')
